### Trích xuất Text gốc từ Link VitePress
Script này sẽ đọc URL VitePress, bóc tách cấu trúc link, tự động tìm file cấu hình `tmc.js` tương ứng, và dò tìm bên trong file Markdown gốc để trả về text chính xác của heading dựa theo cơ chế slugify. Script cũng đã được cập nhật để xuất ra tên bộ kinh (Title) tương ứng từ link.

In [3]:
import urllib.parse
import re
import os
import unicodedata
import json

PROJECT_ROOT = "/Users/ng/projects/n5"

COLLECTION_TITLES = {
    'kinhtuongung': 'Kinh Tương Ưng',
    'kinhtruongbo': 'Kinh Trường Bộ',
    'kinhtrungbo': 'Kinh Trung Bộ',
    'kinhtieubo': 'Kinh Tiểu Bộ',
    'kinhtangchi': 'Kinh Tăng Chi'
}

def slugify(text):
    text = re.sub(r'^[#\s]+', '', text)
    text = text.replace('*', '').replace('**', '')
    text = text.replace('đ', 'd').replace('Đ', 'D').replace('Ð', 'D').replace('ð', 'd')
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.lower()
    text = re.sub(r'[^a-z0-9]+', '-', text)
    text = text.strip('-')
    return text

def get_text_from_link(input_link):
    parsed_url = urllib.parse.urlparse(input_link)
    path_parts = [p for p in parsed_url.path.split('/') if p]
    if not path_parts: return {'title': '', 'heading': ''}

    target_hash = parsed_url.fragment
    collection_key = path_parts[0] if path_parts else ''
    collection_title = COLLECTION_TITLES.get(collection_key, collection_key)

    md_files_to_check = []

    is_compare_link = False
    if len(path_parts) >= 2 and path_parts[-2].startswith('c-'):
        is_compare_link = True

    target_slug = path_parts[-1].replace(".html", "")

    if is_compare_link:
        # Xử lý trường hợp tài liệu được tạo từ tmc.js (dynamic router)
        if len(path_parts) >= 3:
            js_file_path = os.path.join(PROJECT_ROOT, "docs", path_parts[-3], path_parts[-2], "tmc.js")
            if os.path.exists(js_file_path):
                with open(js_file_path, 'r', encoding='utf-8') as f:
                    js_content = f.read()

                pattern = re.compile(r'"slug"\s*:\s*"' + re.escape(target_slug) + r'".*?"left"\s*:\s*"([^"]+)".*?"right"\s*:\s*"([^"]+)"', re.DOTALL)
                match = pattern.search(js_content)

                if match:
                    left_path = match.group(1)
                    right_path = match.group(2)
                    md_files_to_check.extend([
                        os.path.join(PROJECT_ROOT, "docs", left_path.lstrip('/')),
                        os.path.join(PROJECT_ROOT, "docs", right_path.lstrip('/'))
                    ])
    else:
        # Trường hợp tài liệu gốc (có file .md)
        direct_md_path = os.path.join(PROJECT_ROOT, "docs", *path_parts)
        if direct_md_path.endswith('.html'):
            direct_md_path = direct_md_path[:-5]
        direct_md_path += ".md"

        if os.path.exists(direct_md_path):
            md_files_to_check.append(direct_md_path)

    if not md_files_to_check:
        return {'title': collection_title, 'heading': f"Error: No source file found"}

    for md_file_path in md_files_to_check:
        if not os.path.exists(md_file_path):
            continue

        with open(md_file_path, 'r', encoding='utf-8') as f:
            md_content = f.readlines()

        for line in md_content:
            line = line.strip()
            if line.startswith('#'):
                slug = slugify(line)
                if slug == target_hash:
                    clean_heading = line.replace('#', '').replace('*', '').strip()
                    return {'title': collection_title, 'heading': clean_heading}

    return {'title': collection_title, 'heading': f"Error: Hash not found {target_hash}"}


def run_main():
    with open('/Users/ng/projects/n5/.scripts/quotes.md', 'r', encoding='utf-8') as f:
        lines = f.readlines()

    quotes = []
    current_link = None
    current_quote_lines = []

    for line in lines:
        if line.startswith('http://') or line.startswith('https://'):
            if current_link:
                quotes.append({
                    'link': current_link,
                    'quote': '\n'.join(current_quote_lines).strip()
                })
            current_link = line.strip()
            current_quote_lines = []
        else:
            if current_link:
                current_quote_lines.append(line.rstrip('\n'))

    if current_link:
        quotes.append({
            'link': current_link,
            'quote': '\n'.join(current_quote_lines).strip()
        })

    result_array = []
    for q in quotes:
        info = get_text_from_link(q['link'])

        # Format the relative link (remove origin)
        parsed = urllib.parse.urlparse(q['link'])
        rel_link = parsed.path + '#' + parsed.fragment

        result_array.append({
            "quote": q['quote'],
            "title": info['title'],
            "name": info['heading'],
            "link": rel_link
        })

    # Output JS format
    js_output = "[\n"
    for idx, item in enumerate(result_array):
        js_output += "  {\n"
        js_output += f'    "quote": `{item["quote"]}`,\n'
        js_output += f'    "title": "{item["title"]}",\n'
        js_output += f'    "name": "{item["name"]}",\n'
        js_output += f'    "link": "{item["link"]}"\n'
        js_output += "  }"
        if idx < len(result_array) - 1:
            js_output += ",\n"
        else:
            js_output += "\n"
    js_output += "]\n"

    with open('/Users/ng/projects/n5/docs/quotes.config.js', 'w', encoding='utf-8') as f:
        f.write('export default ' + js_output)


run_main()